In [38]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import Window

In [2]:
spark = SparkSession.builder.appName("homework_job").getOrCreate()
spark

25/07/19 21:56:34 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
# setting auto broadcasting off
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print("new broadcast threshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

new broadcast threshold: -1


In [5]:
# reading medals df
medals_df = spark.read.option("header","true").csv("/home/iceberg/data/medals.csv")
maps_df = spark.read.option("header","true").csv("/home/iceberg/data/maps.csv")
medals_matches_players_df = spark.read.option("header","true").csv("/home/iceberg/data/medals_matches_players.csv")
matches_df = spark.read.option("header","true").csv("/home/iceberg/data/matches.csv")
match_details_df = spark.read.option("header","true").csv("/home/iceberg/data/match_details.csv")

In [19]:
# create iceberg table DDL for bucketed tables

# match_details
match_details_bucketed_DDL = """
CREATE TABLE bootcamp.match_details_bucketed (
match_id STRING,
player_gamertag STRING,
previous_spartan_rank STRING,
spartan_rank STRING,
previous_total_xp STRING,
total_xp STRING,
previous_csr_tier STRING,
previous_csr_designation STRING,
previous_csr STRING,
previous_csr_percent_to_next_tier STRING,
previous_csr_rank STRING,
current_csr_tier STRING,
current_csr_designation STRING,
current_csr STRING,
current_csr_percent_to_next_tier STRING,
current_csr_rank STRING,
player_rank_on_team STRING,
player_finished STRING,
player_average_life STRING,
player_total_kills STRING,
player_total_headshots STRING,
player_total_weapon_damage STRING,
player_total_shots_landed STRING,
player_total_melee_kills STRING,
player_total_melee_damage STRING,
player_total_assassinations STRING,
player_total_ground_pound_kills STRING,
player_total_shoulder_bash_kills STRING,
player_total_grenade_damage STRING,
player_total_power_weapon_damage STRING,
player_total_power_weapon_grabs STRING,
player_total_deaths STRING,
player_total_assists STRING,
player_total_grenade_kills STRING,
did_win STRING,
team_id STRING
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(match_details_bucketed_DDL)

# matches
matches_bucketed_DDL = """
CREATE TABLE bootcamp.matches_bucketed (
match_id STRING,
mapid STRING,
is_team_game STRING,
playlist_id STRING,
game_variant_id STRING,
is_match_over STRING,
completion_date STRING,
match_duration STRING,
game_mode STRING,
map_variant_id STRING
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(matches_bucketed_DDL)

# medal_matches_players 
medal_matches_players_bucketed_DDL = """
CREATE TABLE bootcamp.medal_matches_players_bucketed (
match_id STRING,
player_gamertag STRING,
medal_id STRING,
count STRING
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(medal_matches_players_bucketed_DDL)

DataFrame[]

In [28]:
# now save em out
(match_details_df
 .write
 .mode("append")
 .bucketBy(16, "match_id")
 .saveAsTable("bootcamp.match_details_bucketed")
)

(matches_df
 .write
 .mode("append")
 .bucketBy(16, "match_id")
 .saveAsTable("bootcamp.matches_bucketed")
)

(medals_matches_players_df
 .write
 .mode("append")
 .bucketBy(16, "match_id")
 .saveAsTable("bootcamp.medal_matches_players_bucketed")
)

In [29]:
# now save em out
match_details_df_bucketed = spark.read.table("bootcamp.match_details_bucketed")
matches_df_bucketed = spark.read.table("bootcamp.matches_bucketed")
medals_matches_players_df_bucketed = spark.read.table("bootcamp.medal_matches_players_bucketed")

In [30]:
# for some reason, spark plan prints sortmergejoin and not bucketjoin. please analyze my code above and explain whats going on
medals_matches_players_df_bucketed.join(matches_df_bucketed, on="match_id", how="inner").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [match_id#961, player_gamertag#962, medal_id#963, count#964, mapid#942, is_team_game#943, playlist_id#944, game_variant_id#945, is_match_over#946, completion_date#947, match_duration#948, game_mode#949, map_variant_id#950]
   +- SortMergeJoin [match_id#961], [match_id#941], Inner
      :- Sort [match_id#961 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(match_id#961, 200), ENSURE_REQUIREMENTS, [plan_id=357]
      :     +- BatchScan demo.bootcamp.medal_matches_players_bucketed[match_id#961, player_gamertag#962, medal_id#963, count#964] demo.bootcamp.medal_matches_players_bucketed (branch=null) [filters=match_id IS NOT NULL, groupedBy=] RuntimeFilters: []
      +- Sort [match_id#941 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(match_id#941, 200), ENSURE_REQUIREMENTS, [plan_id=358]
            +- BatchScan demo.bootcamp.matches_bucketed[match_id#941, mapid#942, is_team_game#943, playli

In [126]:
joined_df = (
    match_details_df_bucketed
    .join(medals_matches_players_df_bucketed, on=["match_id", "player_gamertag"], how="left")
    .join(matches_df_bucketed, on="match_id", how="inner")
    .join(f.broadcast(maps_df), on="mapid", how="inner")
    .join(
        f.broadcast(
            (medals_df
             .withColumnRenamed("description", "medal_description")
             .withColumnRenamed("name", "medal_name")
            )
        ), on="medal_id", how="left")
)

joined_df.limit(5).toPandas()

,medal_id,mapid,match_id,player_gamertag,previous_spartan_rank,spartan_rank,previous_total_xp,total_xp,previous_csr_tier,previous_csr_designation,...,sprite_left,sprite_top,sprite_sheet_width,sprite_sheet_height,sprite_width,sprite_height,classification,medal_description,medal_name,difficulty
0,3261908037,caacb800-f206-11e4-81ab-24be05e24f7e,000d1f2f-d8b6-4eb2-a517-a835dd3a62c0,BeastModeless,130,130,5485713,5488062,1,6,...,375,525,74,74,1125,899,WeaponProficiency,Kill an opponent by shooting them in the head.,Headshot,60
1,3001183151,caacb800-f206-11e4-81ab-24be05e24f7e,000d1f2f-d8b6-4eb2-a517-a835dd3a62c0,BeastModeless,130,130,5485713,5488062,1,6,...,300,600,74,74,1125,899,Style,Earn the first kill of the match.,First Strike,180
2,3653057799,caacb800-f206-11e4-81ab-24be05e24f7e,000d1f2f-d8b6-4eb2-a517-a835dd3a62c0,BeastModeless,130,130,5485713,5488062,1,6,...,450,750,74,74,1125,899,WeaponProficiency,Kill a player in five shots with the Magnum wi...,Perfect Kill,40
3,2078758684,caacb800-f206-11e4-81ab-24be05e24f7e,000d1f2f-d8b6-4eb2-a517-a835dd3a62c0,BeastModeless,130,130,5485713,5488062,1,6,...,450,300,74,74,1125,899,MultiKill,Kill 2 opponents within 5 seconds of one another.,Double Kill,45
4,285057226,caacb800-f206-11e4-81ab-24be05e24f7e,000d1f2f-d8b6-4eb2-a517-a835dd3a62c0,BeastModeless,130,130,5485713,5488062,1,6,...,75,675,74,74,1125,899,Style,Kill an opponent after you die.,From the Grave,165


In [64]:
# which player averages the most kills per game?
avg_kills_per_game = (match_details_df
 .groupBy("player_gamertag")
 .agg(f.avg("player_total_kills").alias("avg_kills"))
 .withColumn("rank", f.dense_rank().over(Window.orderBy(f.col("avg_kills").desc())))
 .filter(f.col("rank") == 1)
)

avg_kills_per_game.toPandas()

25/07/19 23:33:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:33:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:33:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:33:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:33:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:33:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 2

,player_gamertag,avg_kills,rank
0,gimpinator14,109.0,1


In [78]:
# which playlist gets played the most?
mosted_played_playlist = (matches_df_bucketed
 .groupBy("playlist_id")
 .agg(f.count("playlist_id").alias("num_matches"))
 .withColumn("rank", f.dense_rank().over(Window.orderBy(f.col("num_matches").desc())))
 .filter(f.col("rank") == 1)
)

mosted_played_playlist.toPandas()

25/07/19 23:38:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:38:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:38:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:38:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:38:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:38:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 2

,playlist_id,num_matches,rank
0,f72e0ef0-7c4a-4307-af78-8e38dac3fdba,9350,1


In [85]:
# which map gets played the most?
mosted_played_map = (
    matches_df_bucketed
    .join(f.broadcast(maps_df), on="mapid", how="left")
    .groupBy("name")
    .agg(f.count("*").alias("num_matches"))
    .withColumn("rank", f.dense_rank().over(Window.orderBy(f.col("num_matches").desc())))
    .filter(f.col("rank") == 1)
)

mosted_played_map.limit(10).toPandas()

25/07/19 23:41:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:41:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:41:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:41:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:41:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:41:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 2

,name,num_matches,rank
0,Breakout Arena,8587,1


In [105]:
# which map do players get the most killing spree medals on?
killing_spree_medals = (
    medals_df
    .filter(f.col("name") == "Killing Spree")
    .withColumnRenamed("name", "medal_name")
)

maps_with_most_killing_sprees = (
    medals_matches_players_df_bucketed
    .withColumnRenamed("count", "num_medals")
    .join(f.broadcast(killing_spree_medals), on="medal_id", how="inner")
    .join(matches_df_bucketed.select("match_id", "mapid"), on="match_id", how="inner")
    .join(f.broadcast(maps_df), on="mapid", how="inner")
    .groupBy("name")
    .agg(f.sum("num_medals").alias("num_medals"))
    .withColumn("rank", f.dense_rank().over(Window.orderBy(f.col("num_medals").desc())))
    .filter(f.col("rank") == 1)
)
maps_with_most_killing_sprees.limit(5).toPandas()

25/07/19 23:53:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:53:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:53:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:53:47 WARN DataSourceV2Strategy: Can't translate true to source filter, unsupported expression
25/07/19 23:53:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:53:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/19 23:53:48 WARN WindowExec: No Partition Defined for Window o

,name,num_medals,rank
0,Breakout Arena,6744.0,1


In [128]:
# matches
last_question_DDL = """
CREATE TABLE bootcamp.last_question_v3 (
    medal_id STRING,
    mapid STRING,
    match_id STRING,
    player_gamertag STRING,
    previous_spartan_rank STRING,
    spartan_rank STRING,
    previous_total_xp STRING,
    total_xp STRING,
    previous_csr_tier STRING,
    previous_csr_designation STRING,
    previous_csr STRING,
    previous_csr_percent_to_next_tier STRING,
    previous_csr_rank STRING,
    current_csr_tier STRING,
    current_csr_designation STRING,
    current_csr STRING,
    current_csr_percent_to_next_tier STRING,
    current_csr_rank STRING,
    player_rank_on_team STRING,
    player_finished STRING,
    player_average_life STRING,
    player_total_kills STRING,
    player_total_headshots STRING,
    player_total_weapon_damage STRING,
    player_total_shots_landed STRING,
    player_total_melee_kills STRING,
    player_total_melee_damage STRING,
    player_total_assassinations STRING,
    player_total_ground_pound_kills STRING,
    player_total_shoulder_bash_kills STRING,
    player_total_grenade_damage STRING,
    player_total_power_weapon_damage STRING,
    player_total_power_weapon_grabs STRING,
    player_total_deaths STRING,
    player_total_assists STRING,
    player_total_grenade_kills STRING,
    did_win STRING,
    team_id STRING,
    count STRING,
    is_team_game STRING,
    playlist_id STRING,
    game_variant_id STRING,
    is_match_over STRING,
    completion_date STRING,
    match_duration STRING,
    game_mode STRING,
    map_variant_id STRING,
    name STRING,
    description STRING,
    sprite_uri STRING,
    sprite_left STRING,
    sprite_top STRING,
    sprite_sheet_width STRING,
    sprite_sheet_height STRING,
    sprite_width STRING,
    sprite_height STRING,
    classification STRING,
    medal_description STRING,
    medal_name STRING,
    difficulty STRING
)
USING iceberg
"""

spark.sql(last_question_DDL)

DataFrame[]

In [129]:
# without sortings
joined_df.write.mode("overwrite").saveAsTable("bootcamp.last_question_v3")

In [132]:
%%sql
SELECT SUM(file_size_in_bytes) as size, COUNT(1) AS num_files
FROM demo.bootcamp.last_question_v3.files;

size,num_files
20234258,2


In [135]:
# now sort within partitoins by playlists and maps
joined_df.sortWithinPartitions("playlist_id", "mapid").write.mode("overwrite").saveAsTable("bootcamp.last_question_v3")

In [137]:
%%sql
SELECT SUM(file_size_in_bytes) as size, COUNT(1) AS num_files
FROM demo.bootcamp.last_question_v3.files;

size,num_files
20730266,2


In [139]:
# that didn't change things much probably because the joined df is already pretty sorted
# lets try sorting on a high carindality thing
# now sort within partitoins by playlists and maps
joined_df.sortWithinPartitions("player_total_kills").write.mode("overwrite").saveAsTable("bootcamp.last_question_v3")

In [140]:
%%sql
SELECT SUM(file_size_in_bytes) as size, COUNT(1) AS num_files
FROM demo.bootcamp.last_question_v3.files;

size,num_files
23000166,2


In [142]:
joined_df.sortWithinPartitions("playlist_id", "mapid", "player_gamertag").write.mode("overwrite").saveAsTable("bootcamp.last_question_v3")

In [143]:
%%sql
SELECT SUM(file_size_in_bytes) as size, COUNT(1) AS num_files
FROM demo.bootcamp.last_question_v3.files;

size,num_files
20633842,2
